### Runtime Settings:
* **Accelerator:** 1 GPU with ~12 GB VRAM (RTX 3060 12GB / 3080 / 2080 Ti) — gradient checkpointing is ON to fit.
* **Internet:** On (first run downloads Laya weights ~808 MB)
* **Expected runtime:** ~60–90 min on a 3060 12GB, ~30–45 min on a 3090 / 4090 / A100 (4 epochs over 61k items)

## 1. Environment & GPU Check


In [1]:
!nvidia-smi -L
import torch
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"  GPU 0: {p.name} ({p.total_memory / 1e9:.1f} GB)")
    torch.cuda.set_device(0)
else:
    raise RuntimeError("No CUDA GPU detected. This notebook requires a GPU runtime.")

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# laya README: "If laya.load() hangs: transformers probes for TensorFlow at
# import, and when TF is installed its abseil runtime can deadlock model
# construction. Run with USE_TF=0."
os.environ["USE_TF"] = "0"

GPU 0: NVIDIA GeForce RTX 3060 (UUID: GPU-341c9335-b333-eeaa-8ed1-b3f5ed507c27)
CUDA Available: True | Visible GPUs: 1
  GPU 0: NVIDIA GeForce RTX 3060 (12.5 GB)


## 2. Install Dependencies


In [2]:
# `%pip` targets the kernel's Python; `!pip` may resolve to a different
# interpreter and silently install into the wrong site.
%pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas

import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)

Note: you may need to restart the kernel to use updated packages.
Laya version        : 0.3.20
Transformers version: 5.17.0
PyTorch version     : 2.11.0+cu128


## 3. Load Train Split

Pulls the `train` split only (61,417 rows, stratified on `sentiment_3class`). The test split is **not** loaded in this notebook — it stays untouched for downstream evaluation.

In [3]:
from datasets import load_dataset

DATASET_ID = "shanaka95/fingpt-sentiment-3class"
ds_train = load_dataset(DATASET_ID, split="train")
print(f"Train rows: {len(ds_train)}")
print("Columns   :", ds_train.column_names)
from collections import Counter
print("Label dist:", dict(Counter(ds_train["sentiment_3class"])))
print("\nSample input :", ds_train[0]["input"][:120], "...")
print("Sample gold  :", ds_train[0]["sentiment_3class"])

Train rows: 61417
Columns   : ['input', 'output', 'instruction', 'sentiment_3class']
Label dist: {'positive': 24408, 'neutral': 23372, 'negative': 13637}

Sample input : Its market share widened to 48.51 percent from 48.31 percent a year earlier . ...
Sample gold  : positive


## 4. Question Schema + Build Training Items

Each financial text is wrapped as a single `choice` typed-decisions question with three options (`positive`, `negative`, `neutral`). The gold target is the one-hot distribution over the correct label.

Items are tokenized via `laya.common.build_sequence` and saved to `train_items.pt`. The training script later splits off a 400-item calibration slice (held out from training) with a fixed seed.

In [4]:
import os, json, time, random
import torch
from collections import Counter
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

MODEL_ID = "convaiinnovations/laya"
print(f"Fetching tokenizer + config from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)
print(f"Loaded {MODEL_ID}. max_len={cfg['max_len']} head_max_len={cfg['head_max_len']}")

QUESTIONS = {
    "sentiment": {
        "type": "choice",
        "instructions": (
            "What is the financial sentiment of this text? "
            "Please choose exactly one answer from {positive, negative, neutral}."
        ),
        "criteria": {
            "positive": "The text expresses a positive / bullish financial sentiment.",
            "negative": "The text expresses a negative / bearish financial sentiment.",
            "neutral":  "The text is neutral, factual, or has no clear positive or negative sentiment.",
        },
    }
}

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    seq, markers = build_sequence(
        tok, state,
        {"t": t, "ins": q["instructions"], "crit": crit},
        cfg["max_len"], cfg["head_max_len"],
    )
    if len(markers) != k:
        return None
    return {
        "ids": seq, "markers": markers,
        "qtype": QTYPES[t], "target": target, "label": label,
    }

items = []
skipped = 0
t0 = time.time()
for i, row in enumerate(ds_train):
    state = row["input"]
    gold_label = row["sentiment_3class"]
    gold_q = {"probabilities": {gold_label: 1.0}}
    it = build_training_item(state, QUESTIONS["sentiment"], gold_q)
    if it is None:
        skipped += 1
    else:
        items.append(it)
    if (i + 1) % 10000 == 0:
        print(f"  [{i+1}/{len(ds_train)}] built ({(time.time()-t0):.1f}s)")

print(f"\nBuilt {len(items)} items in {(time.time()-t0):.1f}s ({skipped} skipped)")

ITEMS_PATH = "./train_items.pt"
torch.save(items, ITEMS_PATH)
print(f"Saved to {ITEMS_PATH}")

Fetching tokenizer + config from convaiinnovations/laya...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 38 files:   0%|          | 0/38 [00:00<?, ?it/s]

Loaded convaiinnovations/laya. max_len=512 head_max_len=192
  [10000/61417] built (7.2s)
  [20000/61417] built (14.0s)
  [30000/61417] built (20.7s)
  [40000/61417] built (27.6s)
  [50000/61417] built (34.2s)
  [60000/61417] built (40.8s)

Built 61417 items in 41.7s (0 skipped)
Saved to ./train_items.pt


## 5. Single-GPU Training Script (`train.py`)

Writes the training script to disk, then launches it as a subprocess. The script:
1. Holds out a 400-item calibration slice from training with a fixed seed (same seed across runs → deterministic split).
2. Trains via RLCD for 4 epochs with effective batch 32 (micro=8, accum=4).
3. Fits per-question-type temperatures on the held-out slice.
4. Saves the model in Laya's on-disk format (`model.safetensors` + `encoder/` + `tokenizer/` + `rl_agent_config.json`).

In [5]:
%%writefile train.py
"""Single-GPU RLCD fine-tuning of Laya for 3-class financial sentiment.

Tuned for ~12 GB VRAM (RTX 3060 12GB / 3080 / 2080 Ti):
  - gradient_checkpointing ON (recompute activations, save memory)
  - MICRO_BATCH=16, GRAD_ACCUM=2 -> effective batch 32 (matches 2-GPU DDP per-rank)
  - TF32 matmul + cuDNN benchmark autotune (free speed, no VRAM impact)
  - calibration batch 16

Adapted from the DDP version in laya_finetune_typed_decisions_2xT4_kaggle.ipynb.
"""
import os, sys, time, json, random
import torch
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward


def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids, "attention_mask": att,
        "marker_pos": mpos, "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items]),
    }


def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())


def main():
    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    items_path = sys.argv[3]

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True   # 12 GB VRAM: recompute activations, save memory
    cfg["max_tokens_per_batch"] = 16384    # was 4096; allows MICRO_BATCH=16 * seq 1024
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)

    model.encoder.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    model.head_checkpointing = True

    # Ampere+ speedups: TF32 matmul + cuDNN autotune (free, no VRAM impact)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    device = torch.device("cuda")
    model.to(device)
    model.train()

    all_items = torch.load(items_path, weights_only=False)

    # Same seed + rule as the DDP notebook so the calibration slice is
    # held out deterministically across runs.
    CALIB_MAX = 400
    order = list(range(len(all_items)))
    random.Random(20260922).shuffle(order)
    n_calib = min(CALIB_MAX, len(all_items) // 10)
    calib_items = [all_items[i] for i in sorted(order[:n_calib])]
    train_items = [all_items[i] for i in sorted(order[n_calib:])]

    EPOCHS = 4
    MICRO_BATCH = 16         # was 8; 12 GB gives us ~2x headroom over original 2xT4
    GRAD_ACCUM = 2           # effective batch 32, matches original per-rank
    GROUP_SIZE = 4
    LR_ENCODER = 2.5e-5
    LR_HEAD = 1.0e-4
    SIGMA_START = 0.4
    SIGMA_END = 0.1

    enc_params = [p for n, p in model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD},
    ], weight_decay=0.01)

    total_updates = (len(train_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, total_updates), eta_min=1e-6
    )
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    print(
        f"Starting single-GPU training: {len(train_items)} train items "
        f"({len(calib_items)} held out for calibration) | {EPOCHS} epochs | "
        f"{total_updates} optimizer steps | eff batch {MICRO_BATCH*GRAD_ACCUM}",
        flush=True,
    )
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch)
        random.shuffle(train_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress

        for b_idx in range(0, len(train_items), MICRO_BATCH):
            chunk = train_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            batch = collate_train_batch(chunk, tok.pad_token_id)
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attn      = batch["attention_mask"].to(device, non_blocking=True)
            mpos      = batch["marker_pos"].to(device, non_blocking=True)
            mmask     = batch["marker_mask"].to(device, non_blocking=True)
            qtype     = batch["qtype"].to(device, non_blocking=True)
            target    = batch["target"].to(device, non_blocking=True)

            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = model(input_ids, attn, mpos, mmask, qtype)

            logits = logits.float()
            mask = mmask
            k = mask.sum(-1, keepdim=True).float()

            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)

            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), qtype, mask,
                                  w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)

            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

            scaler.scale(loss).backward()
            accum_step += 1
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(train_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            if (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(
                    f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | "
                    f"Loss: {loss.item()*GRAD_ACCUM:.4f} | "
                    f"Reward: {r.mean().item():.3f} | LR: {cur_lr:.2e}",
                    flush=True,
                )

        print(
            f"=== Epoch {epoch+1}/{EPOCHS} done in {time.time()-t0:.1f}s | "
            f"avg loss: {epoch_loss/max(1, n_batches):.4f} ===",
            flush=True,
        )

    print("\nFitting post-training calibration temperatures...", flush=True)
    del optimizer, scaler, scheduler
    torch.cuda.empty_cache()
    model.eval()

    calib_preds = []
    CALIB_BATCH = 16  # safe at 12 GB with grad checkpointing
    with torch.no_grad():
        for c_idx in range(0, len(calib_items), CALIB_BATCH):
            c_chunk = calib_items[c_idx:c_idx + CALIB_BATCH]
            cb = collate_train_batch(c_chunk, tok.pad_token_id)
            with torch.autocast("cuda", dtype=torch.float16):
                l_sub, _ = model(
                    cb["input_ids"].to(device),
                    cb["attention_mask"].to(device),
                    cb["marker_pos"].to(device),
                    cb["marker_mask"].to(device),
                    cb["qtype"].to(device),
                )
            l_np = l_sub.float().cpu().numpy()
            for r, it in enumerate(c_chunk):
                k = len(it["markers"])
                calib_preds.append((it["qtype"], l_np[r, :k], it["target"]))

    fitted_temps = [1.2, 1.2, 1.2]
    try:
        for qt in range(3):
            sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
            if sel:
                fitted_temps[qt] = fit_one_temp(sel)
        print(
            "Fitted calibration temperatures (choice, score, noul):",
            [round(t, 3) for t in fitted_temps],
            flush=True,
        )
    except Exception as e:
        print("Temperature fitting fallback:", e, flush=True)

    os.makedirs(output_dir, exist_ok=True)
    sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
    save_file(sd, os.path.join(output_dir, "model.safetensors"))
    model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
    tok.save_pretrained(os.path.join(output_dir, "tokenizer"))

    cfg["fine_tuned"] = True
    cfg["model_name"] = "laya-fintiment"
    cfg["temperature"] = fitted_temps
    cfg.pop("temperature_by_options", None)
    with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    print(f"\nModel successfully saved to {output_dir}!", flush=True)


if __name__ == "__main__":
    main()

Writing train.py


## 6. Launch Training (Background)

The full training takes ~30–60 min on A100 / ~60–90 min on T4, which exceeds the cell timeout, so we run it detached. Output streams to `train.log`.

In [7]:
import os, subprocess, sys
# sys.executable (not "python") — the kernel's PATH may not have a bare `python`,
# only `python3`. Using sys.executable guarantees we spawn the same interpreter.
PYTHON = sys.executable

OUTPUT_DIR = "./laya_fintiment"
LOG_PATH = "./train.log"

# If a previous run left a stale log or output dir, clear them so the upload
# step at the end doesn't pick up stale weights.
if os.path.exists(LOG_PATH):
  os.remove(LOG_PATH)

proc = subprocess.Popen(
  [PYTHON, "train.py", model_dir, OUTPUT_DIR, ITEMS_PATH],
  stdout=open(LOG_PATH, "w"),
  stderr=subprocess.STDOUT,
)
print(f"Training launched: PID {proc.pid} | log: {LOG_PATH} | output: {OUTPUT_DIR}")
print(f"Using python: {PYTHON}")
print("Use the next cell to tail progress.")


Training launched: PID 2513 | log: ./train.log | output: ./laya_fintiment
Using python: /venv/main/bin/python
Use the next cell to tail progress.


## 7. Monitor Progress

Tail the log and check whether the process is still alive. Re-run this cell any time.

In [10]:
import os, subprocess
from pathlib import Path

# Find the python pid (the Popen above; ps may show other python procs on the box)
running = False
try:
    poll = proc.poll()
    if poll is None:
        running = True
        print(f"Process {proc.pid} still running (poll=None).")
    else:
        print(f"Process {proc.pid} exited with code {poll}.")
except NameError:
    print("Run the launch cell first.")

log = Path(LOG_PATH)
if log.exists():
    lines = log.read_text().splitlines()
    print(f"\n--- last 30 of {len(lines)} log lines ---")
    for ln in lines[-30:]:
        print(ln)
else:
    print("(no log yet)")

Process 2513 still running (poll=None).

--- last 30 of 3 log lines ---
Starting single-GPU training: 61017 train items (400 held out for calibration) | 4 epochs | 7624 optimizer steps | eff batch 32
/workspace/train.py:185: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


## 8. Wait for Completion


Blocks until the training process exits. If you would rather not block, skip this cell and check `proc.poll()` in the monitor cell above.

In [ ]:
rc = proc.wait()
print(f"Training exited with code {rc}.")
print(f"\n--- final log lines ---")
for ln in Path(LOG_PATH).read_text().splitlines()[-15:]:
    print(ln)

## 9. Verify Saved Model

Sanity-check the on-disk format before pushing to the Hub.

In [ ]:
import os, json
from pathlib import Path

p = Path(OUTPUT_DIR)
print(f"Contents of {OUTPUT_DIR}:")
for child in sorted(p.iterdir()):
    print(f"  {child.name}{'/' if child.is_dir() else ''}")

with open(p / "rl_agent_config.json") as f:
    cfg_saved = json.load(f)
print("\nrl_agent_config.json summary:")
print(f"  fine_tuned = {cfg_saved.get('fine_tuned')}")
print(f"  model_name = {cfg_saved.get('model_name')}")
print(f"  temperature = {cfg_saved.get('temperature')}")

model_size_mb = (p / "model.safetensors").stat().st_size / 1e6
print(f"\nmodel.safetensors size: {model_size_mb:.1f} MB")

## 10. Write Model Card

A short, honest card — no eval numbers since the test split is reserved for the eval notebook.

In [ ]:
README_PATH = os.path.join(OUTPUT_DIR, "README.md")
readme = """---
license: apache-2.0
library_name: transformers
tags:
- laya
- system-one
- calibrated-decisions
- rlcd
- financial-sentiment
- finance
base_model: convaiinnovations/laya
---

# Laya Fine-Tuned for 3-Class Financial Sentiment

[![Base Model](https://img.shields.io/badge/%F0%9F%A4%97%20Base-convaiinnovations%2Flaya-blue)](https://huggingface.co/convaiinnovations/laya)
[![Dataset](https://img.shields.io/badge/%F0%9F%A4%97%20Dataset-shanaka95%2Ffingpt--sentiment--3class-green)](https://huggingface.co/datasets/shanaka95/fingpt-sentiment-3class)

**Laya** ([convaiinnovations/laya](https://huggingface.co/convaiinnovations/laya), 421M params) fine-tuned on the **train split** of [shanaka95/fingpt-sentiment-3class](https://huggingface.co/datasets/shanaka95/fingpt-sentiment-3class) for 3-class financial sentiment classification (`positive` / `negative` / `neutral`).

## Task

Each input (financial news headline or tweet) is wrapped as a typed-decisions `choice` question with three options. The model returns the most likely label as its `choice`, plus calibrated probabilities.

## Training

- **Dataset:** `shanaka95/fingpt-sentiment-3class` train split only (~61k rows; collapsed from the original 9-class FinGPT labels)
- **Method:** RLCD (Reinforcement Learning for Calibrated Decisions) — policy gradients with strictly proper scoring rules, 4 epochs, single GPU
- **Calibration:** 400 items held out from the train split; per-question-type temperature fitting
- **Test split:** untouched (reserved for downstream evaluation)

## Usage

```python
import laya

agent = laya.load(\"shanaka95/laya-fintiment\")

questions = {
    \"sentiment\": {
        \"type\": \"choice\",
        \"instructions\": \"What is the financial sentiment of this text? Choose one from \"\".{positive, negative, neutral}.\".\",
        \"criteria\": {
            \"positive\": \"The text expresses positive / bullish financial sentiment.\",
            \"negative\": \"The text expresses negative / bearish financial sentiment.\",
            \"neutral\":  \"The text is neutral or has no clear sentiment.\",
        },
    }
}

result = agent.predict(\"Stock prices surged on strong earnings.\", questions)
print(result[\"answers\"][\"sentiment\"][\"choice\"])         # -> positive
print(result[\"answers\"][\"sentiment\"][\"probabilities\"])  # -> calibrated probs
```

## License

Apache 2.0. Base model: [convaiinnovations/laya](https://huggingface.co/convaiinnovations/laya).
"""
with open(README_PATH, "w") as f:
    f.write(readme)
print(f"Wrote model card to {README_PATH}")

## 11. Push to Hugging Face Hub

Uses the `hf` CLI (HF_TOKEN read from env). Repo: `shanaka95/laya-fintiment`.

In [ ]:
import os
NEW_REPO = "shanaka95/laya-fintiment"
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set in env. Set it before running this cell."

!hf auth whoami
!hf repos create "{NEW_REPO}" --repo-type model --exist-ok 2>&1 | tail -3 || true
!hf upload "{NEW_REPO}" "./{OUTPUT_DIR}" "." --repo-type model 2>&1 | tail -10